# Week 16: Integrating Agentic AI — Strands Agents & Amazon Bedrock AgentCore

## Learning Objectives

By the end of this session, you will be able to:
1. **Build agents with Strands Agents SDK** — AWS's production agent framework
2. **Create multi-agent systems** using the agents-as-tools pattern
3. **Use AgentCore Memory** for persistent, cross-session agent knowledge
4. **Understand the production stack** — how agents go from notebook to deployment

## Prerequisites

- Completed Week 15 (ReAct agents, LangChain `create_react_agent`, tools, memory)
- Completed Week 13 (Amazon Bedrock Converse API, boto3 setup)
- Watched pre-class videos on multi-agent architectures, orchestration patterns

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Recap | 10 min | Code |
| Section 1: Strands Agents — From LangChain to AWS-Native | 20 min | Demo |
| Lab 1: Build a Fraud Agent with Strands | 15 min | Lab |
| Section 2: Multi-Agent Orchestration | 25 min | Demo |
| Lab 2: Build a Multi-Agent Fraud Pipeline | 15 min | Lab |
| Section 3: AgentCore Memory & Production Services | 15 min | Demo |
| Wrap-up & Homework | 5 min | Markdown |

## The Story So Far

In Week 15, you built your first AI agent — a fraud investigator that could
reason, call tools, and solve multi-step problems. But you built it on your
laptop with LangChain. Now imagine you want to deploy that agent
to handle thousands of fraud cases per day. You need:

- **A production framework** that's AWS-native and battle-tested
- **Persistent memory** so the agent remembers past investigations
- **Multi-agent coordination** so specialized agents collaborate on complex cases
- **Managed infrastructure** so you don't babysit containers

That's exactly what **Strands Agents SDK** + **Amazon Bedrock AgentCore** provide.

## No GPU Needed

All work is API-based through Amazon Bedrock — no GPU required.

# Section 0: Environment Setup

We'll use **Strands Agents SDK** (AWS's open-source agent framework) and
the **Amazon Bedrock AgentCore SDK** for production memory services.

Both work seamlessly with Amazon Bedrock models — no additional API keys needed
beyond your AWS credentials from Week 13.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# strands-agents: AWS's open-source agent framework
# strands-agents-tools: Pre-built tools (calculator, web search, etc.)
# bedrock-agentcore: AgentCore SDK (Memory, Runtime, Gateway)

!pip install -q strands-agents strands-agents-tools bedrock-agentcore

# =============================================================================
# IMPORTS
# =============================================================================
import os
import json
import boto3
from getpass import getpass
from importlib.metadata import version
from strands import Agent, tool
from strands.models import BedrockModel

# AgentCore Memory
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
print("Library versions:")
print(f"  strands-agents:    {version('strands-agents')}")
print(f"  bedrock-agentcore: {version('bedrock-agentcore')}")
print(f"  boto3:             {boto3.__version__}")
print("\nAll libraries installed successfully!")

In [ ]:
# =============================================================================
# AWS CREDENTIALS
# =============================================================================
# Same setup as Week 13 & 15 — your instructor-provided temporary credentials.

if not os.environ.get('AWS_ACCESS_KEY_ID'):
    os.environ['AWS_ACCESS_KEY_ID'] = getpass("AWS Access Key ID: ")
    os.environ['AWS_SECRET_ACCESS_KEY'] = getpass("AWS Secret Access Key: ")
    os.environ['AWS_DEFAULT_REGION'] = input("AWS Region [us-east-1]: ") or 'us-east-1'

# Verify credentials
sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f"✅ Authenticated as: {identity['Arn']}")
print(f"   Region: {os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')}")

# =============================================================================
# MODEL CONFIGURATION
# =============================================================================
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"  # Same as Week 15

# Create Strands BedrockModel
llm = BedrockModel(
    model_id=MODEL_ID,
    region_name=os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'),
)
print(f"   Model: {MODEL_ID}")
print(f"\n💡 Same Claude Haiku model from Weeks 13 & 15 — fast, cheap, supports tool use.")

# Section 1: Strands Agents — From LangChain to AWS-Native

## Framework Comparison

In Week 15, we used **LangChain + LangGraph** to build agents:

```python
# Week 15 (LangChain)
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

llm = ChatBedrockConverse(model="us.anthropic.claude-haiku-4-5-20251001-v1:0")
agent = create_react_agent(llm, tools=[my_tool], checkpointer=MemorySaver())
result = agent.invoke({"messages": [("user", "Investigate TXN-001")]},
                      config={"configurable": {"thread_id": "session-1"}})
```

This week, we switch to **Strands Agents** — AWS's production framework:

```python
# Week 16 (Strands)
from strands import Agent, tool
from strands.models import BedrockModel

llm = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
agent = Agent(model=llm, tools=[my_tool], system_prompt="You are a fraud investigator.")
response = agent("Investigate TXN-001")
```

**Notice**: Same concepts (model, tools, system prompt), cleaner API.

## Why Strands for Production?

| Feature | LangChain (Week 15) | Strands (Week 16) |
|---------|---------------------|-------------------|
| Maintained by | LangChain Inc. | AWS |
| Agent creation | `create_react_agent(llm, tools, ...)` | `Agent(model, tools, ...)` |
| Tool decorator | `@tool` from `langchain_core` | `@tool` from `strands` |
| Memory | `MemorySaver` (in-process) | AgentCore Memory (persistent, managed) |
| Multi-agent | LangGraph StateGraph | Agents-as-tools pattern |
| Deployment | Self-managed | AgentCore Runtime (managed) |
| AWS integration | Via `langchain-aws` adapter | Native |

**Key point**: The CONCEPTS are the same — ReAct loop, tools, memory. What
changes is the framework. This is normal in software: you learn the pattern
once, then apply it in whatever toolkit your company uses.

In [ ]:
# =============================================================================
# DEMO: Rebuild the Fraud Investigation Agent in Strands
# =============================================================================
# In Week 15, we built fraud tools with LangChain's @tool. Now the SAME tools
# in Strands — notice how similar the @tool decorator is.

# --- Tool 1: Look up a transaction ---
@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID from the fraud database.

    Args:
        transaction_id: The transaction ID to look up (e.g., 'TXN-001')
    """
    # Simulated database (same data as Week 15)
    transactions = {
        "TXN-001": {"amount": 4500, "type": "wire_transfer", "merchant": "Unknown Overseas",
                     "time": "3:47 AM", "location": "Lagos, Nigeria",
                     "description": "Wire transfer to unknown overseas account"},
        "TXN-002": {"amount": 89.99, "type": "subscription", "merchant": "Netflix",
                     "time": "6:00 PM", "location": "Chicago, IL",
                     "description": "Monthly streaming subscription"},
        "TXN-042": {"amount": 567.89, "type": "loan_payment", "merchant": "Navient",
                     "time": "12:00 AM", "location": "Auto-debit",
                     "description": "Monthly student loan payment"},
    }
    txn = transactions.get(transaction_id)
    if txn:
        return json.dumps(txn, indent=2)
    return f"Transaction {transaction_id} not found in database."

# --- Tool 2: Check customer history ---
@tool
def check_customer_history(customer_id: str) -> str:
    """Retrieve customer transaction history and behavioral patterns.

    Args:
        customer_id: The customer ID to look up
    """
    histories = {
        "CUST-001": {"avg_transaction": 250, "international_transfers": 0,
                     "account_age_years": 3, "fraud_reports": 0,
                     "typical_hours": "9AM-9PM"},
        "CUST-002": {"avg_transaction": 85, "international_transfers": 0,
                     "account_age_years": 5, "fraud_reports": 0,
                     "typical_hours": "All day"},
    }
    hist = histories.get(customer_id)
    if hist:
        return json.dumps(hist, indent=2)
    return f"Customer {customer_id} not found."

# --- Tool 3: Calculate risk score ---
@tool
def calculate_risk_score(amount: float, is_international: bool,
                         is_unusual_hour: bool, merchant_known: bool) -> str:
    """Calculate fraud risk score based on transaction characteristics.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction crosses borders
        is_unusual_hour: Whether the transaction occurred at an unusual time
        merchant_known: Whether the merchant is in the customer's history
    """
    score = 0
    factors = []
    if amount > 1000:
        score += 30
        factors.append(f"High amount (${amount:,.2f})")
    if is_international:
        score += 25
        factors.append("International transaction")
    if is_unusual_hour:
        score += 20
        factors.append("Unusual hour")
    if not merchant_known:
        score += 25
        factors.append("Unknown merchant")

    level = "LOW" if score < 30 else "MEDIUM" if score < 60 else "HIGH"
    return json.dumps({
        "risk_score": score,
        "risk_level": level,
        "factors": factors,
    }, indent=2)

# --- Tool 4: Check fraud policy ---
@tool
def check_fraud_policy(risk_level: str, amount: float) -> str:
    """Look up the company's fraud policy for a given risk level and amount.

    Args:
        risk_level: Risk level — LOW, MEDIUM, or HIGH
        amount: Transaction amount in dollars
    """
    policies = {
        "LOW": "Auto-approve. Log for monthly audit.",
        "MEDIUM": "Flag for manual review within 24 hours. Notify customer via SMS.",
        "HIGH": "Block transaction immediately. Freeze account. Escalate to fraud team.",
    }
    policy = policies.get(risk_level.upper(), "Unknown risk level")
    escalate = amount > 5000
    return json.dumps({
        "policy": policy,
        "auto_escalate": escalate,
        "reason": f"Amount ${amount:,.2f} exceeds $5,000 threshold" if escalate else "Within limits",
    }, indent=2)

# --- Create the agent ---
fraud_tools = [lookup_transaction, check_customer_history,
               calculate_risk_score, check_fraud_policy]

fraud_agent = Agent(
    model=llm,
    tools=fraud_tools,
    system_prompt=(
        "You are a fraud investigation agent at a financial institution. "
        "When given a transaction to investigate, use your tools to: "
        "1) Look up the transaction details, "
        "2) Check the customer's history, "
        "3) Calculate the risk score, "
        "4) Check the fraud policy. "
        "Then provide a clear verdict with your reasoning."
    ),
)

print("✅ Fraud investigation agent created with Strands!")
print(f"   Model: {MODEL_ID}")
print(f"   Tools: {[t.tool_name for t in fraud_tools]}")
print(f"\n💡 Compare this to Week 15's LangChain agent — same tools, simpler setup.")

In [ ]:
# =============================================================================
# DEMO: Run the Fraud Agent on a Suspicious Transaction
# =============================================================================

print("Investigating TXN-001 (suspected fraud)...")
print("=" * 60)

response = fraud_agent("Investigate transaction TXN-001 for customer CUST-001. "
                       "Determine if this is fraud and what action to take.")

print("=" * 60)
print(f"\nAgent's response:\n{response}")

In [ ]:
# =============================================================================
# DEMO: Same Task, Different Framework — Same Result
# =============================================================================
# The point: agent concepts are PORTABLE. Tools, reasoning, memory — these are
# patterns, not framework-specific features.

print("Framework Comparison:")
print()
print("LangChain (Week 15)                    Strands (Week 16)")
print("-" * 70)
print("from langchain_core.tools import tool   from strands import tool")
print("from langchain_aws import               from strands.models import")
print("    ChatBedrockConverse                     BedrockModel")
print("from langgraph.prebuilt import          agent = Agent(")
print("    create_react_agent                      model=llm,")
print("agent = create_react_agent(                 tools=[...],")
print("    llm, tools=[...],                       system_prompt='...'")
print("    prompt='...'                         )")
print(")")
print()
print("Key differences:")
print("  * Strands: Agent() constructor — all config in one place")
print("  * Strands: Response is direct string, not message dict")
print("  * Strands: @tool decorator auto-parses docstring Args section")
print("  * Both: Same ReAct loop under the hood (think -> act -> observe)")
print("  * Both: Same Bedrock model, same tool-calling protocol")

> **Think About It**: You just rebuilt the SAME fraud agent in a different
> framework — LangChain in Week 15, Strands in Week 16. The tools are identical,
> the model is the same, the reasoning is the same. What does this tell you
> about investing too heavily in one framework? What criteria would you use
> to choose between frameworks for a production system?

## Lab 1: Build a Fraud Agent with Strands (15 minutes)

### Your Task

Build a Strands agent with a NEW tool and a customized system prompt for a
specific fraud investigation scenario.

### Steps

1. **Create a new tool** called `get_similar_transactions` that takes a
   `merchant_name` and returns a list of recent transactions at that merchant.
   Use the `@tool` decorator with full docstring (including Args section).
2. **Create an Agent** with ALL 5 tools (the 4 from the demo + your new one),
   a Bedrock model, and a system prompt that instructs the agent to always
   check for similar transactions before rendering a verdict.
3. **Test your agent** on a transaction at an unknown merchant — does it use
   your new tool?

### Expected Output

- A working `get_similar_transactions` tool
- An Agent that calls all relevant tools
- A verdict that references similar transaction patterns

### Hints

- The `@tool` decorator works exactly like LangChain's — decorate a function,
  add type hints, write a docstring with `Args:` section
- Return a JSON string from your tool (use `json.dumps()`)
- Test with: `agent("Investigate a $3,200 purchase at CryptoExchange for CUST-001")`

### Homework Extension

After class: add a 6th tool that wraps your Week 14 fine-tuned DistilBERT
model as a fraud classifier tool. Load the model with `AutoModelForSequenceClassification`
and expose it as a `@tool` that takes a transaction description and returns
the predicted label + confidence.

In [ ]:
# =============================================================================
# SOLUTION: LAB 1 — BUILD A FRAUD AGENT WITH STRANDS
# =============================================================================

# Step 1: Create the get_similar_transactions tool
@tool
def get_similar_transactions(merchant_name: str) -> str:
    """Find recent transactions at the same merchant across all customers.

    Args:
        merchant_name: The merchant name to search for
    """
    # Simulated database of recent transactions
    all_transactions = {
        "CryptoExchange": [
            {"customer": "CUST-099", "amount": 5000, "flagged": True, "date": "2026-03-20"},
            {"customer": "CUST-145", "amount": 2200, "flagged": True, "date": "2026-03-19"},
            {"customer": "CUST-078", "amount": 800, "flagged": False, "date": "2026-03-18"},
        ],
        "Netflix": [
            {"customer": "CUST-002", "amount": 15.99, "flagged": False, "date": "2026-03-15"},
            {"customer": "CUST-034", "amount": 22.99, "flagged": False, "date": "2026-03-14"},
        ],
        "Unknown Overseas": [
            {"customer": "CUST-201", "amount": 8500, "flagged": True, "date": "2026-03-21"},
        ],
    }

    matches = all_transactions.get(merchant_name, [])
    flagged_count = sum(1 for t in matches if t.get("flagged"))
    return json.dumps({
        "merchant": merchant_name,
        "recent_transactions": matches,
        "total_count": len(matches),
        "flagged_count": flagged_count,
        "risk_note": f"{flagged_count}/{len(matches)} recent transactions at this merchant were flagged"
                     if matches else "No transaction history for this merchant",
    }, indent=2)


# Step 2: Create an Agent with all 5 tools
all_tools = [lookup_transaction, check_customer_history,
             calculate_risk_score, check_fraud_policy,
             get_similar_transactions]

lab1_agent = Agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You are a senior fraud investigator. "
        "When investigating a transaction, ALWAYS: "
        "1) Look up the transaction details, "
        "2) Check the customer's history, "
        "3) Search for similar transactions at the same merchant, "
        "4) Calculate the risk score, "
        "5) Check the fraud policy. "
        "Provide a detailed verdict with evidence from ALL tools."
    ),
)

# Step 3: Test the agent
print("Investigating suspicious crypto transaction...")
print("=" * 60)
test_response = lab1_agent(
    "Investigate a $3,200 purchase at CryptoExchange for customer CUST-001. "
    "Is this fraud?"
)
print("=" * 60)
print(f"\n{test_response}")
print("\n✅ Lab 1 complete!")

# Section 2: Multi-Agent Orchestration

## The Agents-as-Tools Pattern

In production fraud operations, you don't have ONE agent doing everything.
You have **specialized agents** that each handle a piece of the pipeline:

```
+------------------------------------------------------------------+
|                    SUPERVISOR AGENT                                |
|  "Coordinate the fraud investigation — delegate to specialists"   |
|                                                                    |
|  +----------------+  +----------------+  +----------------+       |
|  | Triage Agent   |  | Investigation  |  | Decision       |       |
|  | (classify      |  | Agent          |  | Agent          |       |
|  |  risk level)   |  | (gather        |  | (render        |       |
|  |                |  |  evidence)     |  |  verdict)      |       |
|  +----------------+  +----------------+  +----------------+       |
+------------------------------------------------------------------+
```

**How it works in Strands**:
1. Each specialist is a regular `Agent()` with its own tools and system prompt
2. Wrap each specialist in a `@tool` decorator — now it's callable like a function
3. Give all specialist-tools to a supervisor `Agent()`
4. The supervisor decides which specialists to call and in what order

This is called the **agents-as-tools** pattern. It's simple, powerful, and
the recommended approach in Strands for multi-agent systems.

In [ ]:
# =============================================================================
# DEMO: Define Specialist Agents for the Fraud Pipeline
# =============================================================================
# Each specialist agent has its own tools, system prompt, and expertise.
# We'll then wrap them as tools for a supervisor agent.

# --- Specialist 1: Triage Agent ---
# Fast, rule-based classification. Uses calculate_risk_score tool.
triage_agent = Agent(
    model=llm,
    tools=[lookup_transaction, calculate_risk_score],
    system_prompt=(
        "You are a fraud triage specialist. Your ONLY job is to quickly "
        "classify transactions by risk level. "
        "Look up the transaction, calculate the risk score, and return "
        "a JSON-formatted triage result with: transaction_id, risk_level "
        "(LOW/MEDIUM/HIGH), risk_score, and key_factors. "
        "Be concise — no lengthy analysis. Speed matters in triage."
    ),
    # Suppress streaming output from sub-agents to keep supervisor output clean
    callback_handler=None,
)

# --- Specialist 2: Investigation Agent ---
# Deep dive into suspicious transactions. Uses all evidence-gathering tools.
investigation_agent = Agent(
    model=llm,
    tools=[lookup_transaction, check_customer_history,
           get_similar_transactions],
    system_prompt=(
        "You are a fraud investigation specialist. You receive transactions "
        "that have been flagged as MEDIUM or HIGH risk. Your job is to: "
        "1) Gather all available evidence (transaction details, customer history, "
        "   similar transactions at the same merchant) "
        "2) Write a detailed investigation report with findings "
        "3) Note any red flags or mitigating factors "
        "Be thorough — your report will be used for the final decision."
    ),
    callback_handler=None,
)

# --- Specialist 3: Decision Agent ---
# Renders final verdict based on triage + investigation reports.
decision_agent = Agent(
    model=llm,
    tools=[check_fraud_policy],
    system_prompt=(
        "You are a fraud decision specialist. You receive a triage result "
        "and an investigation report. Your job is to: "
        "1) Check the fraud policy for the given risk level and amount "
        "2) Render a FINAL VERDICT: APPROVE, FLAG_FOR_REVIEW, or BLOCK "
        "3) Provide a one-paragraph justification "
        "4) List the recommended actions "
        "Your decision is final and must be defensible in an audit."
    ),
    callback_handler=None,
)

print("✅ Three specialist agents created:")
print(f"   1. Triage Agent    — tools: lookup_transaction, calculate_risk_score")
print(f"   2. Investigation   — tools: lookup_transaction, check_customer_history, get_similar_transactions")
print(f"   3. Decision Agent  — tools: check_fraud_policy")
print(f"\n💡 callback_handler=None suppresses each agent's intermediate output.")
print(f"   Only the supervisor's output will be visible to the user.")

In [ ]:
# =============================================================================
# DEMO: Wrap Specialist Agents as Tools & Create Supervisor
# =============================================================================
# This is the KEY pattern: each specialist agent becomes a callable tool
# that the supervisor can invoke like a function.

@tool
def run_triage(transaction_id: str, customer_id: str) -> str:
    """Run fraud triage to quickly classify a transaction's risk level.

    Args:
        transaction_id: The transaction ID to triage
        customer_id: The customer ID who made the transaction
    """
    result = triage_agent(
        f"Triage transaction {transaction_id} for customer {customer_id}."
    )
    return str(result)

@tool
def run_investigation(transaction_id: str, customer_id: str,
                      triage_result: str) -> str:
    """Run a deep investigation on a flagged transaction.

    Args:
        transaction_id: The transaction ID to investigate
        customer_id: The customer who made the transaction
        triage_result: The triage result with risk level and factors
    """
    result = investigation_agent(
        f"Investigate transaction {transaction_id} for customer {customer_id}. "
        f"Triage result: {triage_result}"
    )
    return str(result)

@tool
def run_decision(triage_result: str, investigation_report: str,
                 amount: float) -> str:
    """Render a final fraud verdict based on triage and investigation findings.

    Args:
        triage_result: The triage classification result
        investigation_report: The detailed investigation report
        amount: The transaction amount in dollars
    """
    result = decision_agent(
        f"Render a verdict. Triage: {triage_result}. "
        f"Investigation: {investigation_report}. Amount: ${amount:,.2f}."
    )
    return str(result)

# --- Create the Supervisor Agent ---
supervisor = Agent(
    model=llm,
    tools=[run_triage, run_investigation, run_decision],
    system_prompt=(
        "You are the fraud operations supervisor. "
        "You coordinate fraud investigations by delegating to specialist agents: "
        "1) ALWAYS start with run_triage to classify the risk level "
        "2) If risk is MEDIUM or HIGH, run_investigation to gather evidence "
        "3) If risk is LOW, skip investigation and go straight to run_decision "
        "4) ALWAYS end with run_decision for the final verdict "
        "Report the final verdict clearly to the user."
    ),
)

print("✅ Supervisor agent created with 3 specialist agents as tools!")
print(f"   Tools: run_triage, run_investigation, run_decision")
print(f"\n   Flow: Supervisor -> Triage -> (if needed) Investigation -> Decision")

In [ ]:
# =============================================================================
# DEMO: Run the Full Multi-Agent Fraud Pipeline
# =============================================================================

print("Running multi-agent fraud investigation on TXN-001...")
print("=" * 60)

supervisor_response = supervisor(
    "A customer CUST-001 has a suspicious transaction TXN-001 "
    "— a $4,500 wire transfer to an unknown overseas account at 3:47 AM. "
    "Run the full fraud investigation pipeline."
)

print("=" * 60)
print(f"\nSupervisor's final report:\n{supervisor_response}")

In [ ]:
# =============================================================================
# DEMO: Run on a Known-Legitimate Transaction
# =============================================================================
# For a LOW-risk transaction, the supervisor should skip investigation.

print("Running multi-agent investigation on TXN-002 (Netflix subscription)...")
print("=" * 60)

legit_response = supervisor(
    "Investigate transaction TXN-002 for customer CUST-002 "
    "— a $89.99 payment to Netflix."
)

print("=" * 60)
print(f"\nSupervisor's final report:\n{legit_response}")
print(f"\n💡 Notice: for LOW-risk, the supervisor should skip investigation")
print(f"   and go directly to decision. That's intelligent delegation.")

> **Think About It**: The supervisor agent decided whether to skip investigation
> based on the triage risk level. This is an LLM making a routing decision.
> In production, would you trust the LLM to make this routing decision? Or
> would you prefer hard-coded rules (if risk == "LOW": skip investigation)?
> What are the tradeoffs of each approach?

## Lab 2: Build a Multi-Agent Fraud Pipeline (15 minutes)

### Your Task

Extend the multi-agent pipeline with a 4th specialist agent and test the
full pipeline on a new scenario.

### Steps

1. **Create a compliance agent** — a specialist that checks whether a
   transaction violates any regulatory rules (e.g., transactions over $10,000
   require CTR filing, international wire transfers require OFAC screening).
   Give it a `@tool` called `check_compliance_rules`.
2. **Wrap it as a tool** called `run_compliance_check` using the agents-as-tools pattern.
3. **Update the supervisor** — add the compliance tool and update the system
   prompt so the supervisor always runs compliance check after investigation
   but before decision.
4. **Test the full pipeline** on transaction TXN-001 ($4,500 wire transfer
   to overseas). Does the compliance agent flag the OFAC requirement?

### Expected Output

- A working compliance agent with regulatory rules
- Updated supervisor with 4 specialist agents
- A final verdict that includes compliance findings

### Homework Extension

After class: add error handling to the pipeline. What happens if the
triage agent fails or returns an invalid risk level? Implement a fallback
that routes directly to human review when any agent in the pipeline errors.

In [ ]:
# =============================================================================
# SOLUTION: LAB 2 — MULTI-AGENT FRAUD PIPELINE WITH COMPLIANCE
# =============================================================================

# Step 1: Create compliance rules tool
@tool
def check_compliance_rules(amount: float, is_international: bool,
                           destination_country: str) -> str:
    """Check regulatory compliance rules for a transaction.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction crosses borders
        destination_country: The destination country for the transaction
    """
    violations = []
    requirements = []

    # CTR: Currency Transaction Report required for >$10,000
    if amount > 10000:
        requirements.append("CTR (Currency Transaction Report) required — FinCEN regulation")

    # Structuring check: amounts just under $10,000 are suspicious
    if 8000 <= amount <= 10000:
        violations.append("Potential structuring — amount suspiciously close to $10,000 CTR threshold")

    # OFAC screening for international transfers
    high_risk_countries = ["Nigeria", "Russia", "Iran", "North Korea", "Syria"]
    if is_international:
        requirements.append("OFAC screening required for international transfer")
        if destination_country in high_risk_countries:
            violations.append(f"HIGH RISK: {destination_country} is an OFAC-monitored jurisdiction")

    # Wire transfer reporting
    if amount > 3000 and is_international:
        requirements.append("International wire transfer record required (31 CFR 1010.410)")

    return json.dumps({
        "compliant": len(violations) == 0,
        "violations": violations,
        "requirements": requirements,
        "action": "BLOCK — compliance violation" if violations else "PROCEED with requirements",
    }, indent=2)


# Step 2: Create the compliance agent
compliance_agent = Agent(
    model=llm,
    tools=[check_compliance_rules],
    system_prompt=(
        "You are a regulatory compliance specialist. "
        "Check all applicable regulations for the transaction: "
        "BSA/AML, OFAC sanctions, CTR requirements, wire transfer rules. "
        "Return a clear compliance assessment with any violations or requirements."
    ),
    callback_handler=None,
)

# Step 3: Wrap as tool
@tool
def run_compliance_check(amount: float, is_international: bool,
                         destination_country: str) -> str:
    """Run regulatory compliance check on a transaction.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction is international
        destination_country: Destination country
    """
    result = compliance_agent(
        f"Check compliance for a ${amount:,.2f} "
        f"{'international' if is_international else 'domestic'} transaction "
        f"to {destination_country}."
    )
    return str(result)

# Step 4: Create updated supervisor with compliance
updated_supervisor = Agent(
    model=llm,
    tools=[run_triage, run_investigation, run_compliance_check, run_decision],
    system_prompt=(
        "You are the fraud operations supervisor. "
        "Coordinate the full investigation pipeline: "
        "1) ALWAYS start with run_triage to classify risk "
        "2) If risk is MEDIUM or HIGH, run_investigation for evidence "
        "3) ALWAYS run_compliance_check regardless of risk level "
        "4) ALWAYS end with run_decision for the final verdict, "
        "   including compliance findings in your decision context "
        "Report the complete findings to the user."
    ),
)

# Step 5: Test
print("Full pipeline with compliance check on TXN-001...")
print("=" * 60)
test_result = updated_supervisor(
    "Run the full fraud investigation pipeline on transaction TXN-001 "
    "for customer CUST-001. This is a $4,500 international wire transfer "
    "to Nigeria. Include compliance check."
)
print("=" * 60)
print(f"\n{test_result}")
print("\n✅ Lab 2 complete!")

# Section 3: AgentCore Memory — Persistent Agent Knowledge

## From Ephemeral to Persistent Memory

In Week 15, we used LangGraph's `MemorySaver`:
- Simple: `MemorySaver()` + `thread_id`
- **Ephemeral**: Memory dies when the notebook restarts
- **In-process**: Can't share memory between agents or services
- **No long-term learning**: Each session starts from scratch

**Amazon Bedrock AgentCore Memory** solves all of these:
- **Persistent**: Stored in AWS — survives restarts, deployments, scaling
- **Shared**: Any agent (or service) can read/write to the same memory
- **Short-term + Long-term**: Session turns (conversation) + extracted knowledge
- **Searchable**: Semantic search over stored memories
- **Managed**: No database to maintain, automatic scaling

## Memory Types

| Type | What it stores | Use case |
|------|---------------|----------|
| **Short-term** | Turn-by-turn conversation | "What did the user just say?" |
| **Long-term (Semantic)** | Extracted entities and facts | "This customer prefers email notifications" |
| **Long-term (Summary)** | Session summaries | "Last time, we investigated 3 wire transfers" |
| **Long-term (Episodic)** | Structured episodes | "TXN-001 was blocked due to OFAC violation" |

In [ ]:
# =============================================================================
# DEMO: AgentCore Memory — Persistent Fraud Investigation Context
# =============================================================================
# Unlike Week 15's MemorySaver (in-process, ephemeral), AgentCore Memory
# persists across sessions and can be shared between agents.

# NOTE: AgentCore Memory requires a Memory resource to be created first.
# Your instructor has pre-created one for this session.
# In production, you'd create one via console or boto3.

MEMORY_ID = os.environ.get('AGENTCORE_MEMORY_ID', None)

if MEMORY_ID:
    # Create a session manager
    session_manager = MemorySessionManager(
        memory_id=MEMORY_ID,
        region_name=os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'),
    )

    # Create a session for this investigation
    session = session_manager.create_memory_session(
        actor_id="fraud-supervisor",
        session_id="investigation-TXN-001-demo",
    )

    # Record the investigation conversation
    session.add_turns(messages=[
        ConversationalMessage(
            "Investigate transaction TXN-001 — $4,500 wire transfer to Nigeria",
            MessageRole.USER,
        ),
    ])
    session.add_turns(messages=[
        ConversationalMessage(
            "Investigation complete. TXN-001 classified as HIGH risk. "
            "Risk score: 100/100. Factors: high amount, international, "
            "unusual hour, unknown merchant. OFAC screening required. "
            "Verdict: BLOCK. Account frozen pending fraud team review.",
            MessageRole.ASSISTANT,
        ),
    ])

    # Retrieve the conversation
    turns = session.get_last_k_turns(k=5)
    print("AgentCore Memory — Stored Conversation:")
    for turn in turns:
        print(f"  {turn}")

    print(f"\n✅ Investigation context stored in AgentCore Memory!")
    print(f"   Memory ID: {MEMORY_ID}")
    print(f"   Session: investigation-TXN-001-demo")
    print(f"   This persists across notebook restarts and can be accessed by any agent.")
else:
    # Fallback: show the code pattern without live API
    print("AGENTCORE_MEMORY_ID not set — showing code pattern only.")
    print()
    print("In production, you would:")
    print()
    print("  from bedrock_agentcore.memory.session import MemorySessionManager")
    print("  from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole")
    print()
    print("  session_manager = MemorySessionManager(")
    print("      memory_id='your-memory-id',")
    print("      region_name='us-east-1',")
    print("  )")
    print()
    print("  session = session_manager.create_memory_session(")
    print("      actor_id='fraud-supervisor',")
    print("      session_id='investigation-TXN-001',")
    print("  )")
    print()
    print("  session.add_turns(messages=[")
    print("      ConversationalMessage('User message', MessageRole.USER),")
    print("  ])")
    print()
    print("  turns = session.get_last_k_turns(k=5)")
    print()
    print("Key differences from Week 15's MemorySaver:")
    print("   MemorySaver: in-process, dies when notebook restarts")
    print("   AgentCore Memory: persistent, shared, searchable, managed by AWS")

In [ ]:
# =============================================================================
# DEMO: The Full AgentCore Production Stack
# =============================================================================
# Let's map what we've learned to the production architecture.

print("Amazon Bedrock AgentCore — Production Architecture")
print("=" * 60)
print()
print("  +-----------------------------------------------------+")
print("  |                AgentCore Runtime                     |")
print("  |  (Hosts your agents in managed containers)           |")
print("  |                                                      |")
print("  |  +-----------+  +-----------+  +-----------+        |")
print("  |  | Triage    |  | Investigate|  | Decision  |        |")
print("  |  | Agent     |  | Agent     |  | Agent     |        |")
print("  |  | (Strands) |  | (Strands) |  | (Strands) |        |")
print("  |  +-----+-----+  +-----+-----+  +-----+-----+        |")
print("  |        |              |              |               |")
print("  |  +-----+--------------+--------------+-----+        |")
print("  |  |           Supervisor Agent              |        |")
print("  |  +--------------------+--------------------+        |")
print("  +----------------------|------------------------+")
print("                         |")
print("    +-----------+--------+--------+-----------+")
print("    |           |                 |           |")
print("  +-+------+  +-+------+  +------+-+  +------+-+")
print("  | Memory |  |Gateway |  |Identity|  |Observ- |")
print("  |(persis-|  |(MCP    |  |(creds  |  |ability |")
print("  | tent)  |  | tools) |  | mgmt)  |  |(traces)|")
print("  +--------+  +--------+  +--------+  +--------+")
print()
print("What we covered today:")
print("  Strands Agents SDK — build agents (Section 1)")
print("  Multi-agent orchestration — agents-as-tools (Section 2)")
print("  AgentCore Memory — persistent knowledge (Section 3)")
print()
print("What's in the optional notebook:")
print("  AgentCore Runtime — deploy agents to managed containers")
print("  AgentCore Gateway — convert APIs to MCP tools")
print("  AgentCore Identity — secure credential management")
print("  AgentCore Observability — tracing and monitoring")

> **Think About It**: We built a multi-agent fraud pipeline that runs in a
> notebook. In production, this pipeline would need to handle thousands of
> transactions per day, with audit trails, failover, and compliance logging.
> Which AgentCore services (Runtime, Memory, Gateway, Identity, Observability)
> would be most critical to set up first? Why?

## Connecting the Dots: Agents + Fine-Tuned Models

Remember Week 14's fine-tuned DistilBERT fraud classifier? In a production
multi-agent system, you could use it as a **fast pre-filter**:

```python
@tool
def classify_with_distilbert(description: str) -> str:
    """Run the fine-tuned DistilBERT fraud classifier on a transaction."""
    from transformers import pipeline
    classifier = pipeline("text-classification", model="./fraud-classifier")
    result = classifier(description)[0]
    return json.dumps({"prediction": result["label"], "confidence": result["score"]})
```

**The production architecture**:
1. DistilBERT classifies first (FREE, fast, runs locally)
2. Only MEDIUM/HIGH confidence fraud cases go to the full agent pipeline
3. This saves API costs — the LLM agent only handles complex cases

This is a homework extension — try it after class!

## Looking Ahead

- **Weeks 17-18**: RAG (Retrieval-Augmented Generation) — give agents access to
  document knowledge bases. Your AgentCore Gateway can connect to Bedrock
  Knowledge Bases as MCP tools.
- **Weeks 19-20**: MLOps — how to version, monitor, and maintain your agent
  pipelines in production.

# Summary: What We Learned Today

## Key Takeaways

### Strands Agents SDK
- AWS's production agent framework — simpler API than LangChain
- `Agent(model=..., tools=[...], system_prompt="...")` — one clean constructor
- `@tool` decorator works just like LangChain's — concepts are portable
- `from strands.models import BedrockModel` for Bedrock integration

### Multi-Agent Orchestration
- **Agents-as-tools pattern**: wrap specialist agents in `@tool`, give to supervisor
- `callback_handler=None` suppresses sub-agent output for clean supervisor responses
- Supervisor makes routing decisions (skip investigation for LOW risk)
- Each specialist has its own tools and expertise

### AgentCore Memory
- Persistent, cross-session memory (unlike Week 15's ephemeral MemorySaver)
- `MemorySessionManager` -> `create_memory_session` -> `add_turns` / `get_last_k_turns`
- Short-term (conversation) + long-term (semantic, summary, episodic)
- Managed by AWS — no database to maintain

### Production Architecture
- AgentCore = Runtime + Memory + Gateway + Identity + Observability
- Framework-agnostic: works with Strands, LangGraph, CrewAI
- GA since October 2025, available in 9 AWS regions

# Homework and Optional Labs

## Homework (Complete before next session)

### Homework 1: DistilBERT as Agent Tool
Wrap your Week 14 fine-tuned DistilBERT model as a Strands `@tool`.
Build a pipeline where DistilBERT pre-screens transactions and only
sends ambiguous cases to the full multi-agent pipeline.

### Homework 2: Memory-Enriched Agent
If you have access to AgentCore Memory, build an agent that:
- Stores investigation results in AgentCore Memory
- Before investigating a new transaction, searches memory for similar
  past investigations
- Uses past findings to inform current investigation

## Optional Lab: AgentCore Runtime Deployment
See `week_16_optional_agentcore_runtime.ipynb` for a walkthrough of:
- Packaging your Strands agent for AgentCore Runtime
- Deploying with the starter toolkit CLI
- Invoking your deployed agent via boto3
- Setting up AgentCore Gateway for MCP tool integration

# Great Work Today!

You've completed Week 16 of the AI for Data Scientists Academy.

**The journey so far:**
- Weeks 11-12: Prompting LLMs (OpenAI, HuggingFace) — making models respond
- Week 13: Amazon Bedrock — enterprise-grade model access
- Week 14: Fine-tuning — training your own models
- Week 15: Single agents — making models ACT (tools, memory, ReAct)
- **Week 16: Multi-agent systems and production deployment** — making agents
  COLLABORATE at scale with Strands + AgentCore

**Next up**: Weeks 17-18 — RAG (Retrieval-Augmented Generation)